In [21]:
# SVM practice on the sklearn breast cancer dataset before using it on our data
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

In [22]:
# load the dataset (it's built into sklearn so no file needed)
data = load_breast_cancer()

X = data.data      # features -> 30 measurements for each tumour
y = data.target    # labels -> 0 = malignant, 1 = benign

print(X.shape)     # (rows, columns) = (samples, features)
print(pd.Series(y).value_counts())   # how many of each class, not perfectly balanced

(569, 30)
1    357
0    212
Name: count, dtype: int64


In [23]:
# look at a few feature names just to see what the data is
print(data.feature_names[:5])

['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness']


In [24]:
# keep 25% aside for testing, the model never sees it during training
# stratify=y keeps the same benign/malignant ratio in both parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)

print(len(X_train), "training samples")
print(len(X_test), "test samples")

426 training samples
143 test samples


In [25]:
# linear kernel = straight line (hyperplane) between the two classes
model = SVC(kernel="linear", C=1.0)

model.fit(X_train, y_train)        # fit = training, finds the boundary with the biggest gap
y_pred = model.predict(X_test)     # predict = checks which side of the boundary each test sample is on

# compare first few predictions with the real answers
print("predicted:", y_pred[:10])
print("actual:   ", y_test[:10])

acc = accuracy_score(y_test, y_pred)
print("accuracy:", round(acc, 3))

predicted: [1 0 0 1 0 1 0 0 0 1]
actual:    [1 0 0 1 0 1 0 0 0 1]
accuracy: 0.944


In [26]:
# checking how different the feature ranges are
# some features are tiny and some are in the thousands -> the big ones would control the SVM
print("smallest max value:", X_train.max(axis=0).min())
print("largest max value: ", X_train.max(axis=0).max())

smallest max value: 0.02193
largest max value:  4254.0


In [27]:
# StandardScaler -> every feature gets mean 0 and std 1
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # learns mean/std from TRAIN and scales it
X_test_scaled = scaler.transform(X_test)         # uses the same mean/std on TEST (no fitting, otherwise test info leaks in)

model2 = SVC(kernel="linear", C=1.0)
model2.fit(X_train_scaled, y_train)
y_pred2 = model2.predict(X_test_scaled)

# scaling should help
print("without scaling:", round(accuracy_score(y_test, y_pred), 3))
print("with scaling:   ", round(accuracy_score(y_test, y_pred2), 3))

without scaling: 0.944
with scaling:    0.965


In [28]:
# Pipeline puts scaling + model together
# fit() scales using the training data then trains the SVM, predict() scales new data the same way first
# need this for cross validation on the cancer data since the scaler has to be refit on each training fold
pipe = Pipeline([
    ("scaler", StandardScaler()),           # step 1: scale
    ("svm", SVC(kernel="linear", C=1.0)),   # step 2: model
])

pipe.fit(X_train, y_train)        # don't need to scale first, the pipeline does it
pipe_pred = pipe.predict(X_test)

print("pipeline accuracy:", round(accuracy_score(y_test, pipe_pred), 3))   # should match the manual way

pipeline accuracy: 0.965


In [29]:
# confusion matrix -> rows are the real class, columns are what the model predicted
# [[malignant predicted malignant, malignant predicted benign],
#  [benign predicted malignant,    benign predicted benign]]
print(confusion_matrix(y_test, pipe_pred))

[[50  3]
 [ 2 88]]


In [30]:
# trying different C values
# small C -> allows some mistakes, wider margin, simpler model
# big C -> tries to get every training point right, can overfit
# using 5-fold cross validation so the result doesn't depend on one split
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

for c in [0.001, 0.01, 0.1, 1, 10, 100]:
    p = Pipeline([("scaler", StandardScaler()),
                  ("svm", SVC(kernel="linear", C=c))])
    scores = cross_val_score(p, X, y, cv=cv)   # accuracy for each of the 5 folds
    print("C =", c, "-> mean acc:", round(scores.mean(), 3), " folds:", scores.round(2))

# middle values of C did best, very small and very big C were a bit worse
# keeping C = 1.0 fixed for the project for now, tuning later

C = 0.001 -> mean acc: 0.939  folds: [0.93 0.96 0.92 0.93 0.95]
C = 0.01 -> mean acc: 0.97  folds: [0.95 0.99 0.96 0.98 0.96]
C = 0.1 -> mean acc: 0.977  folds: [0.96 0.98 0.97 0.99 0.98]
C = 1 -> mean acc: 0.975  folds: [0.96 0.97 0.98 0.98 0.98]
C = 10 -> mean acc: 0.963  folds: [0.96 0.96 0.97 0.95 0.97]
C = 100 -> mean acc: 0.961  folds: [0.96 0.96 0.96 0.96 0.96]
